In [ ]:
# Script para transformar meu arquivos do BEDtools em dois arquivos FASTA separados, um para ISEScan e outro para BLASTn

# Criando o arquivo do script
nano bedtools_to_fasta.py
# Copiar e colar o código abaixo no arquivo bedtools_to_fasta.py

#####################################################################################################################################################
#!/usr/bin/env python3

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import os
import glob
from pathlib import Path

def parse_gff_line(line):
    """
    Parseia uma linha do arquivo GFF com formato não-padrão 
    O formato tem colunas duplicadas, com informações reais a partir da coluna 10
    """
    fields = line.strip().split('\t')
    
    # Verificar se tem colunas suficientes
    if len(fields) < 18:
        return None
    
    # As informações reais estão nas colunas 10-17 (índices 9-16)
    return {
        'seqid': fields[9],          # coluna 10
        'source': fields[10],         # coluna 11
        'type': fields[11],           # coluna 12
        'start': int(fields[12]),     # coluna 13
        'end': int(fields[13]),       # coluna 14
        'score': fields[14],          # coluna 15
        'strand': fields[15],         # coluna 16
        'phase': fields[16],          # coluna 17
        'attributes': fields[17]      # coluna 18
    }

def extract_id_from_attributes(attributes):
    """Extrai o ID dos atributos do GFF"""
    for attr in attributes.split(';'):
        if attr.startswith('ID='):
            return attr.split('=')[1]
    return None

def find_genome_file(base_name, genome_dir):
    """
    Procura o arquivo do genoma correspondente (.fna ou .fa)
    """
    # Tentar primeiro com .fna
    fna_path = os.path.join(genome_dir, f"{base_name}.fna")
    if os.path.exists(fna_path):
        return fna_path
    
    # Tentar com .fa
    fa_path = os.path.join(genome_dir, f"{base_name}.fa")
    if os.path.exists(fa_path):
        return fa_path
    
    return None

def extract_sequences(gff_file, fasta_file, output_isescan, output_blastn):
    """
    Extrai sequências do genoma baseado nas coordenadas do GFF
    """
    # Carregar o genoma FASTA
    genome = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        genome[record.id] = record.seq
    
    # Listas para armazenar os registros
    isescan_records = []
    blastn_records = []
    
    # Processar o arquivo GFF
    with open(gff_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            
            entry = parse_gff_line(line)
            if not entry:
                continue
            
            # Verificar se é ISEScan ou BLASTn
            source = entry['source']
            if source not in ['ISEScan', 'BLASTn']:
                continue
            
            # Obter a sequência
            seqid = entry['seqid']
            if seqid not in genome:
                continue
            
            # Extrair a subsequência (coordenadas são 1-based no GFF)
            start = entry['start'] - 1  # Converter para 0-based
            end = entry['end']
            seq = genome[seqid][start:end]
            
            # Reverso complementar se necessário
            if entry['strand'] == '-':
                seq = seq.reverse_complement()
            
            # Criar ID descritivo
            seq_id = extract_id_from_attributes(entry['attributes'])
            if not seq_id:
                seq_id = f"{seqid}_{entry['type']}_{start}_{end}"
            
            # Criar descrição
            description = f"{seqid}:{entry['start']}-{entry['end']}({entry['strand']}) {entry['type']} {entry['attributes']}"
            
            # Criar registro SeqRecord
            record = SeqRecord(
                seq,
                id=seq_id,
                description=description
            )
            
            # Adicionar à lista apropriada
            if source == 'ISEScan':
                isescan_records.append(record)
            elif source == 'BLASTn':
                blastn_records.append(record)
    
    # Escrever os arquivos FASTA apenas se houver sequências
    if isescan_records:
        SeqIO.write(isescan_records, output_isescan, "fasta")
    
    if blastn_records:
        SeqIO.write(blastn_records, output_blastn, "fasta")
    
    return len(isescan_records), len(blastn_records)

def process_all_files(gff_dir, genome_dir, output_dir):
    """
    Processa todos os arquivos GFF do diretório
    """
    # Criar diretório de saída se não existir
    os.makedirs(output_dir, exist_ok=True)
    
    # Listar todos os arquivos .gff
    gff_files = glob.glob(os.path.join(gff_dir, "*.gff"))
    
    print(f"Encontrados {len(gff_files)} arquivos GFF para processar\n")
    print("="*80)
    
    # Estatísticas
    total_processed = 0
    total_isescan = 0
    total_blastn = 0
    errors = []
    
    for gff_file in sorted(gff_files):
        # Extrair nome base
        gff_basename = os.path.basename(gff_file)
        
        # Remover sufixo _final.gff ou .gff
        if gff_basename.endswith('_final.gff'):
            base_name = gff_basename.replace('_final.gff', '')
        else:
            base_name = gff_basename.replace('.gff', '')
        
        print(f"\nProcessando: {gff_basename}")
        
        # Procurar arquivo do genoma
        genome_file = find_genome_file(base_name, genome_dir)
        
        if not genome_file:
            error_msg = f"  ❌ Genoma não encontrado para: {base_name}"
            print(error_msg)
            errors.append(error_msg)
            continue
        
        print(f"  ✓ Genoma encontrado: {os.path.basename(genome_file)}")
        
        # Definir nomes dos arquivos de saída
        output_isescan = os.path.join(output_dir, f"{base_name}_isescan.fasta")
        output_blastn = os.path.join(output_dir, f"{base_name}_blastn.fasta")
        
        try:
            # Extrair sequências
            n_isescan, n_blastn = extract_sequences(
                gff_file, 
                genome_file, 
                output_isescan, 
                output_blastn
            )
            
            print(f"  ✓ ISEScan: {n_isescan} sequências → {os.path.basename(output_isescan)}")
            print(f"  ✓ BLASTn: {n_blastn} sequências → {os.path.basename(output_blastn)}")
            
            total_processed += 1
            total_isescan += n_isescan
            total_blastn += n_blastn
            
        except Exception as e:
            error_msg = f"  ❌ Erro ao processar {gff_basename}: {str(e)}"
            print(error_msg)
            errors.append(error_msg)
    
    # Sumário final
    print("\n" + "="*80)
    print("\n📊 RESUMO DO PROCESSAMENTO")
    print("="*80)
    print(f"Total de arquivos processados: {total_processed}/{len(gff_files)}")
    print(f"Total de sequências ISEScan extraídas: {total_isescan}")
    print(f"Total de sequências BLASTn extraídas: {total_blastn}")
    
    if errors:
        print(f"\n⚠️  Erros encontrados: {len(errors)}")
        for error in errors:
            print(error)
    
    print("\n✅ Processamento concluído!")
    print(f"Arquivos salvos em: {output_dir}")

if __name__ == "__main__":
    # Diretórios
    gff_dir = "/mnt/dados/victor_baca/outputs/bedtools"
    genome_dir = "/mnt/dados/victor_baca/genoma"
    output_dir = "/mnt/dados/victor_baca/outputs/cd-hit"
    
    # Executar processamento em batch
    process_all_files(gff_dir, genome_dir, output_dir)
#####################################################################################################################################################

# Instalar Biopython (se necessário)
pip install biopython

# Executar o script
python3 bedtools_to_fasta.py

# Os outputs do script vão ser guardados na pasta /mnt/dados/victor_baca/outputs/cd-hit

In [ ]:
# Ir para o diretório /mnt/dados/victor_baca/outputs/cd-hit, onde estão meus arquivos GFF transformados em FASTA, vou criar duas subpastas para organizar melhor os arquivos FASTA gerados, uma para os arquivos do ISEScan e outra para os arquivos do BLASTn
cd /mnt/dados/victor_baca/outputs/cd-hit
mkdir isescan blastn
mv *_isescan.fasta isescan/
mv *_blastn.fasta blastn/

In [ ]:
# Criando o Script para organizar arquivos FASTA por hospedeiro em subpastas
nano organize_host_files.py
## Os diretórios a serem organizados estão no script abaixo

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import shutil
from pathlib import Path

# Dicionário com mapeamento de genome -> host baseado no genomes.txt
GENOME_HOST_MAP = {
    "GCF_002812445.1": "Fish",
    "GCF_002812465.1": "Fish",
    "GCF_002812505.1": "Fish",
    "GCF_002812425.1": "Fish",
    "GCF_000636115.1": "Fish",
    "GCF_001190805.1": "Fish",
    "GCF_003939065.1": "Fish",
    "GCF_041728555.1": "Fish",
    "GCF_013000945.1": "Fish",
    "GCA_003186145.1": "Fish",
    "GCF_000967445.1": "Fish",
    "GCA_002881395.1": "Fish",
    "GCF_002881775.1": "Fish",
    "GCF_002882835.1": "Fish",
    "GCF_002882745.1": "Fish",
    "GCF_002881695.1": "Fish",
    "GCF_002882645.1": "Fish",
    "GCF_002881615.1": "Fish",
    "GCF_002882555.1": "Fish",
    "GCF_002882465.1": "Fish",
    "GCF_002882375.1": "Fish",
    "GCF_002881595.1": "Fish",
    "GCF_002881575.1": "Fish",
    "GCF_002881555.1": "Fish",
    "GCF_002881535.1": "Fish",
    "GCF_002881515.1": "Fish",
    "GCF_002882275.1": "Fish",
    "GCF_002881495.1": "Fish",
    "GCF_002882205.1": "Fish",
    "GCF_002882125.1": "Fish",
    "GCF_002881475.1": "Fish",
    "GCF_002881455.1": "Fish",
    "GCA_002881435.1": "Fish",
    "GCF_002881415.1": "Fish",
    "GCF_002882035.1": "Fish",
    "GCF_002881935.1": "Fish",
    "GCF_002881375.1": "Fish",
    "GCF_002881355.1": "Fish",
    "GCF_002881335.1": "Fish",
    "GCF_002881315.1": "Fish",
    "GCF_002881865.1": "Fish",
    "GCF_002881295.1": "Fish",
    "GCF_002881255.1": "Fish",
    "GCF_002881215.1": "Fish",
    "GCF_002881195.1": "Fish",
    "GCF_002881235.1": "Fish",
    "GCF_002881175.1": "Fish",
    "GCF_002881155.1": "Fish",
    "GCF_000302475.2": "Fish",
    "SA11-UEL": "Fish",
    "SA10-UEL": "Fish",
    "SA9-UEL": "Fish",
    "SA8-UEL": "Fish",
    "GCF_002197205.1": "Human",
    "GCF_001190885.1": "Human",
    "GCF_001592425.1": "Human",
    "GCF_001592385.1": "Human",
    "GCF_006716245.1": "Human",
    "GCF_001266635.1": "Human",
    "GCF_001552035.1": "Human",
    "GCF_001026925.1": "Human",
    "GCF_000831125.1": "Human",
    "GCF_000831145.1": "Human",
    "GCF_000831105.1": "Human",
    "GCF_025402775.1": "Human",
    "GCF_025402755.1": "Human",
    "GCF_025402735.1": "Human",
    "GCF_021496865.1": "Human",
    "GCF_021496845.1": "Human",
    "GCF_021496825.1": "Human",
    "GCF_021496945.1": "Human",
    "GCF_021496805.1": "Human",
    "GCF_019552345.1": "Human",
    "GCF_030078235.1": "Human",
    "GCF_030078295.1": "Human",
    "GCF_030078315.1": "Human",
    "GCF_030078355.1": "Human",
    "GCF_030078375.1": "Human",
    "GCF_030078255.1": "Human",
    "GCF_030078275.1": "Human",
    "GCF_030078335.1": "Human",
    "GCF_030078395.1": "Human",
    "GCF_030078415.1": "Human",
    "GCF_003966545.1": "Human",
    "GCF_030323365.1": "Human",
    "GCF_021655535.1": "Human",
    "GCF_002289205.1": "Human",
    "GCF_001275545.2": "Human",
    "GCF_002197385.1": "Human",
    "GCF_002197365.1": "Human",
    "GCF_002197325.1": "Human",
    "GCF_002197305.1": "Human",
    "GCF_002197285.1": "Human",
    "GCF_002197245.1": "Human",
    "GCF_002197265.1": "Human",
    "GCF_002197425.1": "Human",
    "GCA_016454805.1": "Human",
    "GCF_030489725.1": "Human",
    "GCF_002214425.1": "Bovine",
    "GCF_000427035.1": "Bovine",
    "GCF_900078265.1": "Bovine",
    "GCF_947313835.2": "Bovine",
    "GCF_029625275.1": "Bovine",
    "GCF_009930895.1": "Bovine",
    "GCF_009930915.1": "Bovine",
    "GCF_029625255.1": "Bovine",
    "GCF_001708205.1": "Bovine",
    "GCF_025631115.1": "Bovine",
    "GCF_025631095.1": "Bovine",
    "GCF_025631125.1": "Bovine",
    "GCF_025801965.1": "Bovine",
    "GCF_027533695.1": "Bovine",
    "GCF_024424535.1": "Bovine",
    "GCF_024424555.1": "Bovine",
    "GCF_024424515.1": "Bovine",
    "GCF_014895495.1": "Bovine",
    "GCA_020980705.1": "Bovine",
    "GCA_020980735.1": "Bovine",
    "GCA_020980665.1": "Bovine",
    "GCF_020980675.1": "Bovine",
    "GCA_020980325.1": "Bovine",
    "GCA_020980385.1": "Bovine",
    "GCF_020980305.1": "Bovine",
    "GCF_020980285.1": "Bovine",
    "GCF_019794075.1": "Bovine",
    "GCF_019794115.1": "Bovine",
    "GCF_019794045.1": "Bovine",
    "GCF_019794085.1": "Bovine",
    "GCF_019794035.1": "Bovine",
    "GCF_019794015.1": "Bovine",
    "GCF_019793995.1": "Bovine",
    "GCF_019793945.1": "Bovine",
    "GCF_019794635.1": "Bovine",
    "GCF_019794875.1": "Bovine",
    "GCF_006543535.1": "Bovine",
    "GCF_006543565.1": "Bovine",
    "GCF_006543545.1": "Bovine",
    "GCF_006543525.1": "Bovine",
    "GCF_006543465.1": "Bovine",
    "GCF_006543275.1": "Bovine",
    "GCF_006543375.1": "Bovine",
    "GCF_006543335.1": "Bovine",
    "GCF_006543305.1": "Bovine",
    "GCF_006543265.1": "Bovine",
    "GCF_006543115.1": "Bovine",
    "GCF_006543125.1": "Bovine",
    "GCF_006543135.1": "Bovine",
    "GCF_003605605.1": "Bovine",
    "GCA_020981825.1": "Bovine",
    "GCA_020981865.1": "Bovine",
    "GCA_020981795.1": "Bovine",
    "GCA_020981845.1": "Bovine",
    "GCA_020981705.1": "Bovine",
    "GCA_020981685.1": "Bovine",
    "GCA_020980445.1": "Bovine",
    "GCA_020980475.1": "Bovine",
    "GCA_020980425.1": "Bovine",
    "GCF_020980405.1": "Bovine",
    "GCA_020980155.1": "Bovine",
}

def get_genome_id(filename):
    """Extrai o ID do genoma do nome do arquivo."""
    # Remove extensões e sufixos
    base = filename.replace("_blastn.fasta", "").replace("_isescan.fasta", "")
    
    # Casos especiais para amostras SA
    if base.startswith("SA"):
        return base
    
    # Extrai o ID do genoma (ex: GCF_002812445.1)
    parts = base.split("_")
    if len(parts) >= 2:
        genome_id = f"{parts[0]}_{parts[1]}"
        return genome_id
    
    return base

def organize_files(base_dir, dry_run=False):
    """
    Organiza os arquivos por hospedeiro em subpastas.
    
    Args:
        base_dir: Diretório base contendo os arquivos
        dry_run: Se True, apenas mostra o que seria feito sem copiar
    """
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"❌ Diretório não encontrado: {base_dir}")
        return
    
    # Criar subpastas para cada hospedeiro
    hosts = ["Fish", "Human", "Bovine"]
    for host in hosts:
        host_dir = base_path / host
        if not dry_run:
            host_dir.mkdir(exist_ok=True)
        print(f"📁 Subpasta criada/verificada: {host_dir}")
    
    # Listar arquivos .fasta
    fasta_files = list(base_path.glob("*.fasta"))
    
    stats = {"Fish": 0, "Human": 0, "Bovine": 0, "Unknown": 0}
    unknown_files = []
    
    print(f"\n🔍 Processando {len(fasta_files)} arquivos em {base_dir}...\n")
    
    for file in fasta_files:
        genome_id = get_genome_id(file.name)
        host = GENOME_HOST_MAP.get(genome_id, "Unknown")
        
        if host != "Unknown":
            dest_dir = base_path / host
            dest_file = dest_dir / file.name
            
            if not dry_run:
                shutil.copy2(file, dest_file)
                print(f"✅ {file.name} → {host}/")
            else:
                print(f"[DRY RUN] {file.name} → {host}/")
            
            stats[host] += 1
        else:
            stats["Unknown"] += 1
            unknown_files.append(file.name)
            print(f"⚠️  {file.name} → Hospedeiro desconhecido")
    
    # Resumo
    print(f"\n{'='*60}")
    print(f"📊 RESUMO - {base_dir}")
    print(f"{'='*60}")
    print(f"🐟 Fish:   {stats['Fish']} arquivos")
    print(f"👤 Human:  {stats['Human']} arquivos")
    print(f"🐄 Bovine: {stats['Bovine']} arquivos")
    print(f"❓ Unknown: {stats['Unknown']} arquivos")
    print(f"{'='*60}\n")
    
    if unknown_files:
        print("⚠️  Arquivos com hospedeiro desconhecido:")
        for f in unknown_files:
            print(f"   - {f}")
        print()

def main():
    """Função principal."""
    print("="*60)
    print("🧬 ORGANIZADOR DE ARQUIVOS POR HOSPEDEIRO")
    print("="*60)
    print()
    
    # Diretórios a processar
    dirs = [
        "/mnt/dados/victor_baca/outputs/cd-hit/blastn",
        "/mnt/dados/victor_baca/outputs/cd-hit/isescan"
    ]
    
    # Perguntar se quer fazer um dry run primeiro
    print("Deseja executar em modo de teste (dry run) primeiro?")
    print("Isso mostrará o que será feito sem copiar os arquivos.")
    response = input("Digite 's' para dry run ou 'n' para executar: ").strip().lower()
    
    dry_run = response == 's'
    
    if dry_run:
        print("\n🔍 MODO DRY RUN ATIVADO - Nenhum arquivo será copiado\n")
    
    # Processar cada diretório
    for dir_path in dirs:
        organize_files(dir_path, dry_run=dry_run)
        print()
    
    if dry_run:
        print("✅ Dry run concluído! Execute novamente sem dry run para copiar os arquivos.")
    else:
        print("✅ Organização concluída com sucesso!")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# Tornar o script executável
chmod +x organize_host_files.py

# Executar
python organize_host_files.py